In [ ]:
from comet_ml import API

api_key = "PUT_API_KEY"
workspace = "PUT_WORKSPACE"
project_name = "sampling-bench"

filter_name = "funnel_10"

api = API(api_key=api_key)

experiments = api.get_experiments(workspace=workspace, project_name=project_name)

filtered_experiments = []
for experiment in experiments:
    if filter_name in experiment.name:
        filtered_experiments.append(experiment)

In [128]:
import pandas as pd
metric_name_to_table_name = {"KL/eubo": "EUBO", "KL/elbo": "ELBO", "logZ/delta_reverse": "$\Delta \log Z$", "Z/delta_reverse": "$\Delta Z$", "discrepancies/sd": "SD"}

def get_target_sd(experiment):
    for metric in experiment.get_metrics():
        if metric["metricName"] == "discrepancies/sd_target":
            return float(metric["metricValue"])

def get_metrics(experiment, steps):
    result = {}
    for metric in experiment.get_metrics():
        if metric["metricName"] not in metric_name_to_table_name.keys():
            continue
        if metric["step"] in steps:
            continue
        result[metric_name_to_table_name[metric["metricName"]]] = metric["metricValue"]
    algorithm_name = [param["valueCurrent"] for param in experiment.get_parameters_summary() if param["name"] == "algorithm_name"][0]
    algorithm_name = algorithm_name.replace("_", "\_")
    result["algorithm_name"] = "$\\text{" + algorithm_name + "}$"
    return result

# Collect all metric dicts with algorithm name as one entry
rows = []
for experiment in filtered_experiments:
    row = get_metrics(experiment, [19999, 20000])
    rows.append(row)

target_sd = get_target_sd(filtered_experiments[0])

df = pd.DataFrame(rows)
df = df.set_index("algorithm_name").rename_axis("Algorithm")
df = df.replace("NaN", "-")
for col in df.columns:
    df[col] = df[col].apply(lambda x: round(float(x), 3) if isinstance(x, (float, int)) or (isinstance(x, str) and x.replace('.', '', 1).replace('-', '', 1).isdigit()) else x)

print(df, end="\n\n\n")

latex_table = df.to_latex(
    escape=False,
    float_format=lambda x: f"{x:.3f}",
    label=f"table:{filter_name}",
    caption=None,
)

for param in filtered_experiments[0].get_parameters_summary():
    if param["name"] == "algorithm_name":
        algorithm_name = param["valueCurrent"]
        env_name = filtered_experiments[0].name.split(f"{algorithm_name}_", 1)[-1] if f"{algorithm_name}_" in filtered_experiments[0].name else ""
        break
env_name = env_name.replace("_", "\_")
caption_text = f"{env_name}. Target SD {target_sd:.3f}."
caption_text = "\\caption{" + caption_text + "}\n"
latex_table = latex_table.replace("\\begin{table}", "\\begin{table}[!h]\n\\centering")
latex_table = latex_table.replace("\\end{table}", caption_text + "\\end{table}")
print(latex_table)

                    EUBO $\Delta Z$       SD   ELBO $\Delta \log Z$
Algorithm                                                          
$\text{gfn\_tb}$       -          -        -      -               -
$\text{pis}$      27.068     -0.082  144.594 -0.464          -0.086
$\text{dds}$       2.861     -0.078  136.854 -0.487          -0.081
$\text{dis}$      31.035      0.299  118.564 -0.834           0.262


\begin{table}[!h]
\centering
\label{table:funnel_10}
\begin{tabular}{llllll}
\toprule
 & EUBO & $\Delta Z$ & SD & ELBO & $\Delta \log Z$ \\
Algorithm &  &  &  &  &  \\
\midrule
$\text{gfn\_tb}$ & - & - & - & - & - \\
$\text{pis}$ & 27.068 & -0.082 & 144.594 & -0.464 & -0.086 \\
$\text{dds}$ & 2.861 & -0.078 & 136.854 & -0.487 & -0.081 \\
$\text{dis}$ & 31.035 & 0.299 & 118.564 & -0.834 & 0.262 \\
\bottomrule
\end{tabular}
\caption{funnel\_10. Target SD 119.463.}
\end{table}

